# ETS MARL — Data Science Analysis

Implements the analyses described in `planning/data_science_analysis_plan.md`
(v3). Every section ties back to a specific thesis claim. The structure is:

| § | Analysis | Claim | Methods |
|---|---|---|---|
| C1 | Convergence | RQ1.a | Rolling-mean plateau + PELT change-point |
| C2 | Cross-seed reproducibility | RQ1.b | Fan charts + cross-seed CV |
| S1 | Strategy archetypes | RQ3.a | KMeans + hierarchical + PCA |
| S2 | Reward-weight effect on strategy | RQ3.b | S1 pipeline applied to all-financial variant; centroid contrast |
| S3 | Economic rationality of strategies | RQ3.c | LASSO logit + Random Forest + anchor-vs-realised price |
| R1 | Regulatory shifts | RQ2.a | Cluster shift in shared PCA space + outcome boxplots (IQR) |
| R2 | UDBC compliance pathways | RQ2.b | Multinomial logit + RF + SMOTE; transition heatmaps |

**Data ingestion.** Mirrors `notebooks/ets_marl - Default RQ Analysis.ipynb`
and `notebooks/ets_marl - Sweep Analysis.ipynb`: load one or more sweep specs,
expand each variant's overrides over the base config, then walk
`<output_dir>/<variant>/` for `training_log_*.csv` (per episode) and
`year_log_*.csv` (per year). All runs are keyed by `(sweep, variant, seed)`.

**Robustness.** Every section guards against missing data — sections that
need a non-default variant (S2 cross-variant, R1, R2 cross-variant) skip
gracefully when only default seeds are loaded. This means the notebook
runs end-to-end on the smoke dataset
(`configs/sweeps/smoke_ds_analysis.yaml`) and on the full sweep without
edits.

**Methodology.** Standardisation before any distance-based method (KMeans,
PCA, LASSO). Train/val splits or cross-validation for any predictive model.
Multi-method robustness on every headline claim — see the table in the
planning doc §7.


In [ ]:
# Install notebook dependencies if needed.
# This notebook imports scikit-learn, scipy, ruptures, imbalanced-learn etc.,
# so the cell auto-installs `notebooks/requirements-notebooks.txt` whenever any
# of those modules is missing. Using `subprocess` + `sys.executable` avoids the
# pitfall where `%pip install -r "{PROJECT_ROOT/...}"` fails because IPython
# magics don't expand Python `{var}` placeholders at runtime.
import subprocess, sys, importlib
from pathlib import Path

INSTALL_NOTEBOOK_DEPS = False  # flip to True to force-reinstall

def _find_requirements():
    req_name = "requirements-notebooks.txt"
    here = Path.cwd()
    candidates = [here / req_name, here / "notebooks" / req_name]
    candidates.extend(p / "notebooks" / req_name for p in here.parents)
    return next((p for p in candidates if p.exists()), None)

def _missing_ds_dep():
    # Any one of these missing -> trigger install. sklearn/imblearn are the
    # ones users typically hit on a fresh kernel.
    for mod in ("sklearn", "imblearn", "ruptures", "scipy", "seaborn"):
        try:
            importlib.import_module(mod)
        except ImportError:
            return mod
    return None

req_path = _find_requirements()
missing = _missing_ds_dep()
if INSTALL_NOTEBOOK_DEPS or missing is not None:
    if req_path is None:
        raise FileNotFoundError("Could not find notebooks/requirements-notebooks.txt")
    if missing is not None:
        print(f"`{missing}` not found in this kernel — installing notebook deps from {req_path}")
    else:
        print(f"Force-installing notebook dependencies from: {req_path}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", str(req_path)])
else:
    print("All notebook dependencies (sklearn, imblearn, ruptures, scipy, seaborn) found; skipping install.")
    print("(Set INSTALL_NOTEBOOK_DEPS = True to force a reinstall.)")


## 1. Setup — Locate Project Root

In [2]:
import os, sys
from pathlib import Path

def _is_project_root(path: str) -> bool:
    return os.path.isdir(os.path.join(path, "src")) and os.path.isdir(os.path.join(path, "configs"))

_candidates = ["..", ".", os.path.join("..", ".."),
               os.path.join("Thesis-Energy-Auction", "ets_marl_happo_current"), "Thesis-Energy-Auction"]
_found = _is_project_root(".")
if not _found:
    for _c in _candidates:
        if _is_project_root(_c):
            os.chdir(_c); _found = True; break
if not _found:
    raise RuntimeError(f"Could not locate project root from {os.getcwd()}.")

PROJECT_ROOT = Path(os.getcwd()).resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print("Project root:", PROJECT_ROOT)

Project root: C:\Users\danie\Documents\Python_Scripts\Master Thesis\Thesis-Energy-Auction


## 2. Imports

Standard scientific stack plus a few sklearn / specialised modules
used by the analyses below.

In [3]:
import re
import yaml
import warnings
import itertools
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ML / DS toolbox
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition   import PCA
from sklearn.cluster         import KMeans, AgglomerativeClustering
from sklearn.metrics         import silhouette_score, classification_report
from sklearn.linear_model    import LogisticRegression
from sklearn.ensemble        import RandomForestClassifier
from sklearn.inspection      import partial_dependence
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline        import Pipeline
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster

# Specialised — change-point detection (justified in methodology) and
# minority-class oversampling for R2.
import ruptures as rpt
try:
    from imblearn.over_sampling import SMOTE
    _HAS_SMOTE = True
except ImportError:
    _HAS_SMOTE = False

# Project utilities — sweep loader and price anchor.
from src.utils.sweep        import load_sweep_spec, expand_dotted_overrides, deep_merge
from src.utils.price_anchor import compute_fundamental_anchor

sns.set_style('whitegrid')
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning, module='sklearn')

RNG = np.random.default_rng(0)
print('Imports OK. SMOTE available:', _HAS_SMOTE)

ModuleNotFoundError: No module named 'sklearn'

## 3. Sweep Specs to Load

List one or more sweep specs (paths relative to the project root). Each
spec's `output_dir` may be overridden if results were moved. Multiple
specs can be combined — runs from every spec are merged into a single
`(sweep, variant, seed)`-keyed dict.

To validate the notebook end-to-end on a tiny dataset, run
`python scripts/sweep.py --spec configs/sweeps/smoke_ds_analysis.yaml`
first, then keep `SWEEP_SPECS` pointed at it.

To analyse a real run, swap in the production sweep spec(s) (e.g.
`configs/sweeps/default_seeds.yaml`).

In [ ]:
SWEEP_SPECS = [
    PROJECT_ROOT / 'configs' / 'sweeps' / 'default_seeds.yaml',
    # PROJECT_ROOT / 'configs' / 'sweeps' / 'smoke_ds_analysis.yaml',
]
# Per the user's actual on-disk layout, sweep CSVs live directly in
# `results/sweeps/default/` (no per-variant subfolder besides agent
# checkpoint dirs). Override here so we don't need to move files.
OUTPUT_DIR_OVERRIDES = [
    PROJECT_ROOT / 'results' / 'sweeps' / 'default',
    # None,
]
assert len(SWEEP_SPECS) == len(OUTPUT_DIR_OVERRIDES)

# Convergence-window definition. ``TAIL_FRAC`` of the longest run is used
# for steady-state metrics; this is overridden per-run by the C1 detector
# when both PELT and the rolling-plateau detector agree.
TAIL_FRAC = 0.20

## 4. Discover & Load Runs

For every `(variant, seed)` pair we resolve the per-episode
(`training_log_*.csv`) and per-year (`year_log_*.csv`) logs. The sweep
launcher writes them with a `--run-tag <variant>` infix; older
single-config runs do not, so we try the tagged filename first and fall
back to the untagged one. Variants whose `overrides` deviate from the
base config are tagged with `has_overrides=True` so downstream sections
can filter for the default-only sub-population.

In [ ]:
def _resolve_run_files(variant_name, seed, variant_dir, output_dir):
    """Return (training_log, year_log) paths or (None, None). Tagged then untagged."""
    search_dirs = [d for d in (variant_dir, output_dir) if d is not None]
    training_names = [f"training_log_{variant_name}_s{seed}.csv", f"training_log_s{seed}.csv"]
    year_names     = [f"year_log_{variant_name}_s{seed}.csv",     f"year_log_s{seed}.csv"]
    tlog = ylog = None
    for base in search_dirs:
        if tlog is None:
            for n in training_names:
                p = base / n
                if p.exists():
                    tlog = p; break
        if ylog is None:
            for n in year_names:
                p = base / n
                if p.exists():
                    ylog = p; break
    return tlog, ylog


def _load_runs_from_spec(spec_path, output_override):
    spec = load_sweep_spec(str(spec_path))
    base_cfg_path = (PROJECT_ROOT / spec['base_config']).resolve()
    with open(base_cfg_path) as f:
        base_cfg = yaml.safe_load(f)
    output_dir = Path(output_override) if output_override else (PROJECT_ROOT / spec['output_dir']).resolve()
    sweep_name = Path(spec_path).stem

    runs = {}
    for v in spec['variants']:
        seeds = v.get('seeds') or spec.get('seeds') or []
        overrides = v.get('overrides') or {}
        has_overrides = bool(overrides)
        eff_cfg = deep_merge(base_cfg, expand_dotted_overrides(overrides))
        variant_dir = output_dir / v['name']
        for s in seeds:
            tlog, ylog = _resolve_run_files(v['name'], s, variant_dir, output_dir)
            if tlog is None:
                continue
            ep_df = pd.read_csv(tlog)
            yr_df = pd.read_csv(ylog) if ylog is not None else pd.DataFrame()
            runs[(sweep_name, v['name'], int(s))] = {
                'sweep': sweep_name, 'variant': v['name'], 'seed': int(s),
                'has_overrides': has_overrides, 'overrides': overrides,
                'config': eff_cfg, 'ep_df': ep_df, 'yr_df': yr_df,
                'training_log': str(tlog), 'year_log': str(ylog) if ylog else None,
            }
    return runs


runs = {}
for spec_path, override in zip(SWEEP_SPECS, OUTPUT_DIR_OVERRIDES):
    runs.update(_load_runs_from_spec(spec_path, override))

if not runs:
    raise RuntimeError('No runs found. Re-check SWEEP_SPECS / OUTPUT_DIR_OVERRIDES.')

print(f'Loaded {len(runs)} run(s).')
for k, r in runs.items():
    n_ep = len(r['ep_df']); n_yr = len(r['yr_df'])
    flag = '[overrides]' if r['has_overrides'] else '[default]'
    print(f"  {k}  ep={n_ep:>5}  yr={n_yr:>5}  {flag}")

## 5. Shared Helpers

These wrap the column-naming convention used throughout the trainer
(`<metric>_A<i>` for agent-level columns, 1-indexed) and provide a few
analysis-window primitives reused across sections.

In [ ]:
# Number of learning agents and timeline length, taken from the first run.
_first_run = next(iter(runs.values()))
N_AGENTS = int(_first_run['config']['companies']['n_agents'])
N_YEARS  = int(_first_run['config']['simulation']['n_years'])
print(f'N_AGENTS={N_AGENTS}  N_YEARS={N_YEARS}')


def agent_cols(df, prefix, max_idx=None):
    """Return ``[<prefix>_A1, ..., <prefix>_AN]`` for columns present in ``df``."""
    if max_idx is None: max_idx = N_AGENTS
    rx = re.compile(rf'^{re.escape(prefix)}_A(\d+)$')
    out = []
    for c in df.columns:
        m = rx.match(c)
        if m and int(m.group(1)) <= max_idx:
            out.append((int(m.group(1)), c))
    return [c for _, c in sorted(out)]


def archetype_of(agent_idx_0based: int, config: dict) -> str:
    """Designed archetype label from initial mix (% green = onshore + offshore + solar)."""
    mix = config['companies']['initial_mix'][agent_idx_0based]
    green = float(mix[2] + mix[3] + mix[4])
    if green < 0.50:  return 'coal-leaning'
    if green < 0.60:  return 'balanced'
    if green < 0.70:  return 'transitioner'
    return 'green-leader'


def weighting_of(agent_idx_0based: int, config: dict) -> str:
    w = config['companies']['reward_weights'][agent_idx_0based]
    if float(w[1]) <= 1e-6:  return 'pure_financial'
    return 'balanced_esg'


def converged_window(ep_df, tail_frac=TAIL_FRAC, override_start=None):
    """Return the steady-state slice of an episode log."""
    n = len(ep_df)
    if override_start is not None and 0 <= override_start < n:
        return ep_df.iloc[override_start:].copy()
    cut = max(1, int(np.floor(n * (1.0 - tail_frac))))
    return ep_df.iloc[cut:].copy()


def tail_year_rows(run, conv_starts=None):
    """year_log rows from the converged window, filtered by the per-run cutoff."""
    yr = run['yr_df']
    if yr.empty: return yr
    n_ep = len(run['ep_df'])
    if conv_starts is not None and run_key(run) in conv_starts:
        start = conv_starts[run_key(run)]
    else:
        start = max(0, int(np.floor(n_ep * (1.0 - TAIL_FRAC))))
    return yr[yr['episode'] >= start].copy()


def run_key(run):
    return (run['sweep'], run['variant'], run['seed'])


# Convenience: split runs by default vs override.
DEFAULT_RUNS = {k: v for k, v in runs.items() if not v['has_overrides']}
OVERRIDE_RUNS = {k: v for k, v in runs.items() if v['has_overrides']}
print(f'default-config runs: {len(DEFAULT_RUNS)}   override-config runs: {len(OVERRIDE_RUNS)}')
print('variants present:', sorted({k[1] for k in runs}))

---

## §C1 — Convergence (claim RQ1.a)

> Two detectors on episode-level system reward and `clearing_price_last`:
> a curriculum-native rolling-mean plateau check, and the specialised
> PELT change-point. We use the **later** of the two as the conservative
> convergence cutoff per run, and flag detector disagreement.

The cutoff is then propagated to every steady-state metric in §C2–§S3.

In [ ]:
def _system_reward_series(ep_df):
    cols = agent_cols(ep_df, 'reward')
    return ep_df[cols].sum(axis=1).values if cols else np.array([])


def rolling_plateau(series, window=None, tol_rel=0.02, hold_windows=3):
    """First episode where the rolling mean stays within ``tol_rel`` of the
    final-window mean for ``hold_windows`` consecutive windows."""
    n = len(series)
    if n < 20: return None
    if window is None:
        window = max(20, n // 20)
    s = pd.Series(series).rolling(window, min_periods=window).mean().values
    final_mean = np.nanmean(s[-window:])
    if not np.isfinite(final_mean) or abs(final_mean) < 1e-9:
        band = max(1.0, abs(final_mean) * tol_rel + 1.0)
    else:
        band = abs(final_mean) * tol_rel
    in_band = np.abs(s - final_mean) <= band
    streak = 0
    for i, ok in enumerate(in_band):
        if not np.isfinite(s[i]):
            streak = 0; continue
        streak = streak + 1 if ok else 0
        if streak >= hold_windows * window:
            return max(0, i - hold_windows * window + 1)
    return None


def pelt_change(series, pen=None):
    """PELT change-point on an L2 cost; returns the last change-point or None."""
    n = len(series)
    if n < 20: return None
    x = np.asarray(series, dtype=float)
    x = x[np.isfinite(x)]
    if len(x) < 20: return None
    if pen is None:
        pen = 3.0 * np.log(len(x)) * np.var(x)
    algo = rpt.Pelt(model='l2', min_size=max(5, len(x) // 50)).fit(x)
    cps = algo.predict(pen=pen)
    cps = [c for c in cps if c < len(x)]
    return cps[-1] if cps else None


conv_results = []
conv_starts = {}
for k, run in runs.items():
    rew = _system_reward_series(run['ep_df'])
    px  = run['ep_df']['clearing_price_last'].values if 'clearing_price_last' in run['ep_df'].columns else np.array([])
    plat_r = rolling_plateau(rew); plat_p = rolling_plateau(px)
    pelt_r = pelt_change(rew);     pelt_p = pelt_change(px)
    candidates = [c for c in (plat_r, plat_p, pelt_r, pelt_p) if c is not None]
    cutoff = max(candidates) if candidates else max(1, len(run['ep_df']) // 2)
    conv_starts[k] = cutoff
    conv_results.append({
        'run': f'{k[1]}/s{k[2]}', 'n_ep': len(run['ep_df']),
        'plateau_reward': plat_r, 'plateau_price': plat_p,
        'pelt_reward':    pelt_r, 'pelt_price':    pelt_p,
        'convergence_cutoff': cutoff,
        'tail_frac': round(1.0 - cutoff / max(1, len(run['ep_df'])), 3),
    })
conv_df = pd.DataFrame(conv_results)
display(conv_df)

In [ ]:
# Visualise reward and price trajectories with the agreed cutoff marked.
n = len(runs)
fig, axes = plt.subplots(n, 2, figsize=(14, 3.0 * n), squeeze=False)
for row_idx, (k, run) in enumerate(runs.items()):
    rew = _system_reward_series(run['ep_df'])
    px  = run['ep_df'].get('clearing_price_last', pd.Series([])).values
    cutoff = conv_starts[k]
    for ax, y, ylabel in [(axes[row_idx, 0], rew, 'system reward'),
                          (axes[row_idx, 1], px,  'clearing price (last yr)')]:
        if len(y) == 0:
            ax.set_axis_off(); continue
        ax.plot(y, lw=0.6, alpha=0.5, color='#888', label='raw')
        smooth = pd.Series(y).rolling(max(5, len(y)//30), min_periods=1).mean()
        ax.plot(smooth, lw=1.6, color='C0', label='rolling mean')
        ax.axvline(cutoff, color='red', ls='--', lw=1.2, label=f'cutoff @ ep {cutoff}')
        ax.set(xlabel='episode', ylabel=ylabel, title=f'{k[1]}/s{k[2]}')
        ax.legend(fontsize=8, loc='lower right')
plt.tight_layout(); plt.show()

**Interpretation.** The two detectors should *broadly* agree on
where the trajectory levels off; we take the later of the two as the
conservative cutoff. With a 100-episode smoke run the detectors are
noisy by construction (insufficient samples for a stable rolling mean
or PELT prior), so the cutoff lands near the run end — that is
expected. On the production-scale sweep these cutoffs typically settle
around 60–80 % of `n_episodes`.

---

## §C2 — Cross-Seed Reproducibility (claim RQ1.b)

> Seed-bin fan charts on six headline metrics + a converged-window CV
> table. If converged-window CV is small relative to expected
> cross-variant differences, RQ2 / RQ3 claims are well-supported.

In [ ]:
def _system_metric(df, agg, prefix=None, col=None):
    if prefix is not None:
        cols = agent_cols(df, prefix)
        if not cols: return pd.Series(dtype=float)
        x = df[cols]
        return x.sum(axis=1) if agg == 'sum' else x.mean(axis=1)
    if col is not None and col in df.columns:
        return df[col]
    return pd.Series(dtype=float)


HEADLINE = [
    ('system_reward',        lambda r: _system_metric(r['ep_df'], 'sum',  prefix='reward')),
    ('quality_score',        lambda r: _system_metric(r['ep_df'], 'sum',  col='quality_score')),
    ('clearing_price_last',  lambda r: _system_metric(r['ep_df'], 'sum',  col='clearing_price_last')),
    ('mean_green_frac',      lambda r: _system_metric(r['ep_df'], 'mean', prefix='green_frac')),
    ('mean_penalty',         lambda r: _system_metric(r['ep_df'], 'mean', prefix='penalty')),
    ('mean_invest_cost',     lambda r: _system_metric(r['ep_df'], 'mean', prefix='invest_cost')),
]

# Operate within the default-only runs; cross-variant fan charts belong
# in R1, not the credibility section.
default_keys = list(DEFAULT_RUNS)

def _fan_panel(ax, name, fn, smoothing=20):
    series = []
    for k in default_keys:
        s = fn(DEFAULT_RUNS[k])
        if s.empty: continue
        series.append(s.rolling(smoothing, min_periods=1).mean().values)
    if not series:
        ax.set_axis_off(); return
    L = min(len(s) for s in series)
    arr = np.stack([s[:L] for s in series])
    med = np.median(arr, axis=0)
    lo, hi = np.percentile(arr, [16, 84], axis=0)
    ax.fill_between(np.arange(L), lo, hi, alpha=0.25, color='C0', label='±1σ across seeds')
    ax.plot(med, color='C0', lw=1.6, label='median')
    for s in arr:
        ax.plot(s, color='C0', lw=0.4, alpha=0.4)
    ax.set(title=name, xlabel='episode')
    ax.legend(fontsize=8)


fig, axes = plt.subplots(2, 3, figsize=(18, 8))
for ax, (name, fn) in zip(axes.flat, HEADLINE):
    _fan_panel(ax, name, fn)
plt.suptitle('§C2 — Cross-seed fan charts (default-config runs)', y=1.02, fontsize=13)
plt.tight_layout(); plt.show()

In [ ]:
# Converged-window CV table.
rows = []
for name, fn in HEADLINE:
    vals = []
    for k in default_keys:
        s = fn(DEFAULT_RUNS[k])
        if s.empty: continue
        ep = DEFAULT_RUNS[k]['ep_df']
        cut = conv_starts.get(k, int(0.8 * len(ep)))
        seg = s.iloc[cut:]
        if len(seg) > 0:
            vals.append(float(seg.mean()))
    if not vals: continue
    arr = np.array(vals)
    mu = float(arr.mean()); sd = float(arr.std(ddof=1)) if len(arr) > 1 else float('nan')
    rows.append({'metric': name, 'n_seeds': len(arr),
                 'mean across seeds': mu, 'std across seeds': sd,
                 'cv (%)': 100.0 * sd / abs(mu) if mu else float('nan')})
cv_df = pd.DataFrame(rows)
display(cv_df.style.format({'mean across seeds': '{:.4g}', 'std across seeds': '{:.3g}', 'cv (%)': '{:.2f}'}))

**Interpretation.** On the production sweep we want headline `cv (%)`
below ~10 %. On the smoke dataset (only 2 seeds × 100 episodes) the CVs
are large and uninterpretable — the table is shown as a sanity check
that the pipeline runs, not as evidence.

---

## §S1 — Strategy Archetypes in the Converged Market (claim RQ3.a)

Per-(run, agent) feature vector built from the converged window. We
cluster with KMeans (k chosen by **Silhouette + Elbow**), confirm with
hierarchical clustering (Ward linkage, dendrogram), visualise in 2D via
PCA, and cross-tabulate the emergent clusters against the **designed**
archetype × reward-weighting cells. The clustering pipeline produced
here is reused by §S2 (variant comparison) and §R1 (regulatory shifts).

In [ ]:
# --- Per-(run, agent) feature matrix --------------------------------
EPISODE_FEATURES = ['bid_price', 'avg_bid_mult', 'inv_onshore_share',
                    'inv_offshore_share', 'inv_solar_share', 'green_frac',
                    'invest_cost', 'penalty',
                    # v8.6.1 episode-level credit/debt aggregates
                    # (silently NaN-filled on older logs that lack them).
                    'peak_loan_outstanding', 'peak_carry_forward',
                    'final_treasury_reserve']
YEAR_FEATURES    = ['sec_buy_vol', 'sec_sell_vol',
                    # v8.6.1 year-level compliance/credit features
                    'coverage_gap', 'effective_penalty_rate',
                    'carry_forward_end', 'treasury_reserve',
                    'loan_outstanding']
UDBC_BUCKETS     = ['U', 'D', 'M', 'B', 'C']


def build_strategy_features(run_subset, conv_starts):
    rows = []
    for k, run in run_subset.items():
        ep = run['ep_df']
        yr = run['yr_df']
        cut = conv_starts.get(k, int(0.8 * len(ep)))
        ep_c = ep.iloc[cut:]
        yr_c = yr[yr['episode'] >= cut] if 'episode' in yr.columns else yr
        cfg = run['config']
        for i in range(N_AGENTS):
            row = {'sweep': k[0], 'variant': k[1], 'seed': k[2],
                   'agent': i + 1,
                   'archetype': archetype_of(i, cfg),
                   'weighting': weighting_of(i, cfg)}
            for prefix in EPISODE_FEATURES:
                col = f'{prefix}_A{i+1}'
                row[prefix] = float(ep_c[col].mean()) if col in ep_c.columns else np.nan
            for prefix in YEAR_FEATURES:
                col = f'{prefix}_A{i+1}'
                row[prefix] = float(yr_c[col].mean()) if col in yr_c.columns else np.nan
            udbc_total = 0.0; udbc_vec = {}
            for b in UDBC_BUCKETS:
                col = f'udbc_{b}_total_A{i+1}'
                v = float(ep_c[col].mean()) if col in ep_c.columns else 0.0
                udbc_vec[b] = v; udbc_total += v
            for b in UDBC_BUCKETS:
                row[f'udbc_{b}_share'] = (udbc_vec[b] / udbc_total) if udbc_total > 1e-9 else 0.0
            rows.append(row)
    return pd.DataFrame(rows)


def feature_columns(df):
    base = list(EPISODE_FEATURES) + list(YEAR_FEATURES) + [f'udbc_{b}_share' for b in UDBC_BUCKETS]
    return [c for c in base if c in df.columns]


feat_df = build_strategy_features(DEFAULT_RUNS, conv_starts)
print(f'Per-(run, agent) feature matrix: {feat_df.shape[0]} rows × {feat_df.shape[1]} cols')
display(feat_df.head())

In [ ]:
# --- Standardise + KMeans with Silhouette/Elbow ---------------------
def _prepare_X(df):
    feats = feature_columns(df)
    X = df[feats].values.astype(float)
    # Drop columns that are entirely NaN or zero-variance.
    keep = []
    for j, name in enumerate(feats):
        col = X[:, j]
        col = col[np.isfinite(col)]
        if len(col) > 0 and np.nanstd(col) > 1e-12:
            keep.append(j)
    feats = [feats[j] for j in keep]
    X = X[:, keep]
    # Replace NaN with column means before scaling.
    col_means = np.nanmean(X, axis=0)
    nan_mask = ~np.isfinite(X)
    X[nan_mask] = np.take(col_means, np.where(nan_mask)[1])
    scaler = StandardScaler().fit(X)
    return scaler.transform(X), feats, scaler


X, feats, scaler = _prepare_X(feat_df)
print('Feature columns kept after variance filter:', feats)
print('X shape:', X.shape)

K_RANGE = range(2, min(8, len(X)))
sil_scores, inertias = [], []
for k in K_RANGE:
    if k >= len(X):
        sil_scores.append(np.nan); inertias.append(np.nan); continue
    km = KMeans(n_clusters=k, n_init=10, random_state=0).fit(X)
    inertias.append(km.inertia_)
    try:
        sil_scores.append(silhouette_score(X, km.labels_))
    except Exception:
        sil_scores.append(np.nan)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(list(K_RANGE), inertias, 'o-')
axes[0].set(title='Elbow — KMeans inertia', xlabel='k', ylabel='inertia')
axes[1].plot(list(K_RANGE), sil_scores, 'o-', color='C2')
axes[1].set(title='Silhouette — KMeans', xlabel='k', ylabel='silhouette')
plt.tight_layout(); plt.show()

valid = [(k, s) for k, s in zip(K_RANGE, sil_scores) if np.isfinite(s)]
K_BEST = max(valid, key=lambda kv: kv[1])[0] if valid else 3
print(f'Selected k = {K_BEST} (max silhouette).')

In [ ]:
# --- Final KMeans + hierarchical confirmation -----------------------
km = KMeans(n_clusters=K_BEST, n_init=20, random_state=0).fit(X)
feat_df = feat_df.copy()
feat_df['kmeans_cluster'] = km.labels_

# Hierarchical (Ward) — confirms cluster count via dendrogram.
Z = linkage(X, method='ward')
hier_labels = fcluster(Z, t=K_BEST, criterion='maxclust')
feat_df['hier_cluster'] = hier_labels

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
labels = [f"{r['variant'][:6]}/s{r['seed']}/A{r['agent']}" for _, r in feat_df.iterrows()]
dendrogram(Z, labels=labels, leaf_rotation=90, ax=axes[0])
axes[0].set(title=f'Hierarchical clustering (Ward) — cut at k={K_BEST}')
axes[0].tick_params(axis='x', labelsize=7)

# PCA scatter coloured by KMeans cluster.
pca = PCA(n_components=2, random_state=0).fit(X)
P = pca.transform(X)
ax = axes[1]
for c in range(K_BEST):
    sel = km.labels_ == c
    ax.scatter(P[sel, 0], P[sel, 1], label=f'cluster {c}', s=60, alpha=0.85)
ax.set(title=f'PCA projection — KMeans clusters (var explained: {pca.explained_variance_ratio_.sum():.2%})',
       xlabel=f'PC1 ({pca.explained_variance_ratio_[0]:.1%})',
       ylabel=f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
# --- Cross-tabulate emergent cluster vs designed archetype × weighting
ct = pd.crosstab(
    [feat_df['archetype'], feat_df['weighting']],
    feat_df['kmeans_cluster'],
    margins=True,
)
print('Designed archetype × weighting  vs  KMeans cluster:')
display(ct)

# Same for hierarchical for the multi-method robustness story.
ct_h = pd.crosstab(
    [feat_df['archetype'], feat_df['weighting']],
    feat_df['hier_cluster'],
    margins=True,
)
print('Designed archetype × weighting  vs  Hierarchical cluster:')
display(ct_h)

**Interpretation.** A diagonal-ish cross-tab indicates the
emergent clustering recovers the designed archetypes. Off-diagonal mass
is itself a finding (e.g. archetype heterogeneity collapsing under a
specific reward weighting, or two archetypes converging onto the same
strategy). The hierarchical and KMeans tables should agree on the
qualitative grouping — when they don't, the strategy space is
ambiguous and the headline claim needs hedging.

In [ ]:
# Persist the fitted feature space so §S2 and §R1 can project new
# (variant, seed, agent) feature vectors into the *same* space.
S1_ARTIFACTS = dict(scaler=scaler, pca=pca, kmeans=km, feature_cols=feats, k=K_BEST)
print('Stored S1 artefacts: scaler, PCA, KMeans, feature_cols.')

---

## §S2 — How Reward Weights Shape Strategy (claim RQ3.b)

This is the headline RQ3 analysis. We use **two complementary contrasts**:

1. **Within the default config:** every default run already mixes
   pure-financial agents (`reward_weights=[1,0]`) with balanced-ESG
   agents (`reward_weights=[0.5,0.5]`). We project them into the §S1
   PCA space and compare centroid positions and within-group spread.
2. **Cross-variant (when available):** if the loaded sweeps include an
   `all_financial`-style variant (every learning agent with
   `[w_cost=1, w_green=0]`), we re-run the §S1 clustering on that
   variant's per-agent feature matrix and compare cluster count, spread
   in the §S1 PCA basis, and silhouette score against the default.

In [ ]:
# --- (1) Within-default contrast ------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
for w, color in [('pure_financial', 'C3'), ('balanced_esg', 'C2')]:
    sel = feat_df['weighting'] == w
    if sel.sum():
        ax.scatter(P[sel, 0], P[sel, 1], label=w, color=color, s=70, alpha=0.85, edgecolor='k', linewidth=0.4)
        cx, cy = P[sel, 0].mean(), P[sel, 1].mean()
        ax.scatter([cx], [cy], marker='X', s=260, color=color, edgecolor='black', linewidth=1.5,
                   label=f'{w} centroid')
ax.set(title='Within-default — pure-financial vs balanced-ESG agents in §S1 PCA',
       xlabel=f'PC1 ({pca.explained_variance_ratio_[0]:.1%})',
       ylabel=f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
ax.legend(fontsize=8)

# Per-weighting feature means (standardised) — what differentiates the two?
ax = axes[1]
mu = feat_df.groupby('weighting')[feats].mean()
mu_std = (mu - mu.mean()) / (mu.std(ddof=0) + 1e-9)
mu_std.T.plot(kind='bar', ax=ax, color=['C2', 'C3'])
ax.axhline(0, color='k', lw=0.5)
ax.set(title='Standardised feature means by weighting',
       ylabel='z-score (across weighting groups)', xlabel='')
ax.legend(title='weighting')
plt.tight_layout(); plt.show()

In [ ]:
# --- (2) Cross-variant contrast (skip gracefully if unavailable) ----
financial_variants = [k for k, v in OVERRIDE_RUNS.items()
                      if 'all_financial' in v['variant'] or 'financial' in v['variant']]
if not financial_variants:
    print('No all-financial variant loaded — cross-variant S2 skipped.')
else:
    af_runs = {k: OVERRIDE_RUNS[k] for k in financial_variants}
    af_feat = build_strategy_features(af_runs, conv_starts)
    # Project AF rows into the *default-fitted* feature space for a like-for-like comparison.
    af_X = af_feat[feats].values.astype(float)
    col_means = np.nanmean(af_X, axis=0)
    nan_mask = ~np.isfinite(af_X)
    af_X[nan_mask] = np.take(col_means, np.where(nan_mask)[1])
    af_X_std = scaler.transform(af_X)
    af_P = pca.transform(af_X_std)

    # Cluster the AF feature matrix on its own (independent KMeans + silhouette).
    af_K_RANGE = range(2, min(8, len(af_X_std)))
    af_sil = []
    for k in af_K_RANGE:
        if k >= len(af_X_std):
            af_sil.append(np.nan); continue
        af_km = KMeans(n_clusters=k, n_init=10, random_state=0).fit(af_X_std)
        try:
            af_sil.append(silhouette_score(af_X_std, af_km.labels_))
        except Exception:
            af_sil.append(np.nan)
    af_valid = [(k, s) for k, s in zip(af_K_RANGE, af_sil) if np.isfinite(s)]
    af_k = max(af_valid, key=lambda kv: kv[1])[0] if af_valid else K_BEST
    af_km_final = KMeans(n_clusters=af_k, n_init=20, random_state=0).fit(af_X_std)

    # Two-panel PCA: default vs all-financial in the SAME projection.
    fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharex=True, sharey=True)
    for ax, points, labels, title in [
        (axes[0], P, km.labels_, f'default — KMeans (k={K_BEST})'),
        (axes[1], af_P, af_km_final.labels_, f'all-financial — KMeans (k={af_k})'),
    ]:
        for c in np.unique(labels):
            sel = labels == c
            ax.scatter(points[sel, 0], points[sel, 1], s=60, alpha=0.85, label=f'cluster {c}')
        ax.set(title=title, xlabel='PC1', ylabel='PC2')
        ax.legend(fontsize=8)
    plt.suptitle('§S2 — Strategy space: default vs all-financial (§S1 PCA basis)', y=1.02)
    plt.tight_layout(); plt.show()

    # Numerical comparison: spread and silhouette.
    def _spread(P_): return float(np.mean(np.linalg.norm(P_ - P_.mean(axis=0), axis=1)))
    summary = pd.DataFrame([
        {'variant': 'default',        'n': len(P),    'silhouette': max(sil_scores) if sil_scores else np.nan,
         'k_best': K_BEST, 'mean radius (PC1-PC2)': _spread(P)},
        {'variant': 'all_financial',  'n': len(af_P), 'silhouette': max(af_sil) if af_sil else np.nan,
         'k_best': af_k, 'mean radius (PC1-PC2)': _spread(af_P)},
    ])
    display(summary)
    print('Cluster-count and spread comparison drives the RQ3.b headline claim.')

**Interpretation.** The within-default contrast is the lower-bar
evidence: even *inside* the same training run, balanced-ESG agents
should sit further along whichever PC tracks green investment than
their pure-financial twins. The cross-variant comparison is the
stronger test: if the all-financial variant collapses to fewer
clusters and a tighter PCA cloud, removing the ESG component is
*causing* strategic homogenisation. If the spread does **not**
collapse, agents found other reasons to differentiate (cost-structure
heterogeneity, urgency scalars, etc.) — a finding worth reporting
honestly.

---

## §S3 — Are Strategies Economically Rational? (claim RQ3.c, Tier 2)

Three sub-analyses:

* **H1.** *Carbon price drives green investment.* Per-(run, agent, year):
  label = 1 if `invest_cost > threshold`. Features = lag-1 clearing price,
  archetype dummies, agent bank at start of year, year. Logistic
  regression with **L1/LASSO** (Lecture 4 + 8) and a **Random Forest**
  for the partial-dependence cross-check on the lagged carbon price.
* **H2.** *High-fossil agents transition faster.* Label = 1 if the
  agent's `delta_green` over the episode exceeds the median. Features =
  initial mix, archetype dummies, mean clearing price seen by the
  agent. Same LASSO + RF setup.
* **Anchor check.** Realised mean clearing price per year (across runs)
  vs `compute_fundamental_anchor(year, config)`. One overlay plot,
  Pearson correlation, and mean absolute deviation.

In [ ]:
# --- (a) Anchor-vs-realised credibility plot -------------------------
def _per_year_clearing(run):
    yr = run['yr_df']
    if yr.empty or 'year' not in yr.columns: return None
    cut = conv_starts.get(run_key(run), int(0.8 * len(run['ep_df'])))
    yrc = yr[yr['episode'] >= cut] if 'episode' in yr.columns else yr
    return yrc.groupby('year')['clearing_price'].mean()


anchor_rows = []
for k, run in DEFAULT_RUNS.items():
    cp = _per_year_clearing(run)
    if cp is None: continue
    for y in range(N_YEARS):
        anchor_rows.append({
            'sweep': k[0], 'variant': k[1], 'seed': k[2],
            'year': y,
            'clearing': float(cp.get(y, np.nan)),
            'anchor':   compute_fundamental_anchor(y, run['config']),
        })
anchor_df = pd.DataFrame(anchor_rows)
panel = anchor_df.groupby('year').agg(realised_mean=('clearing', 'mean'),
                                      realised_std =('clearing', 'std'),
                                      anchor=('anchor', 'mean')).reset_index()

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(panel['year'], panel['anchor'],        '-o',  color='black',      lw=2,   label='fundamental anchor')
ax.plot(panel['year'], panel['realised_mean'], '-s',  color='C0',         lw=2,   label='realised mean (across seeds)')
ax.fill_between(panel['year'],
                panel['realised_mean'] - panel['realised_std'].fillna(0),
                panel['realised_mean'] + panel['realised_std'].fillna(0),
                alpha=0.20, color='C0', label='±1σ across seeds')
ax.set(xlabel='year', ylabel='EUR / tCO₂', title='§S3 anchor check — realised clearing vs fundamental anchor')
ax.legend()
plt.tight_layout(); plt.show()

valid = panel.dropna(subset=['realised_mean', 'anchor'])
if len(valid) > 1:
    corr = float(np.corrcoef(valid['realised_mean'], valid['anchor'])[0, 1])
    mad  = float(np.mean(np.abs(valid['realised_mean'] - valid['anchor'])))
    print(f'Pearson correlation (realised, anchor): {corr:+.3f}')
    print(f'Mean absolute deviation: {mad:.2f} EUR/t')
else:
    print('Not enough year-level data points to compute correlation.')

In [ ]:
# --- (b) Hypothesis test datasets ------------------------------------
def _per_agent_year_table(runs_dict):
    rows = []
    for k, run in runs_dict.items():
        yr = run['yr_df']
        cfg = run['config']
        if yr.empty or 'year' not in yr.columns:
            continue
        cut = conv_starts.get(k, int(0.8 * len(run['ep_df'])))
        yrc = yr[yr['episode'] >= cut] if 'episode' in yr.columns else yr
        # Year-mean clearing price across the converged window.
        cp_year = yrc.groupby('year')['clearing_price'].mean().to_dict()
        for y in range(N_YEARS):
            for i in range(N_AGENTS):
                inv_col = f'invest_cost_A{i+1}'
                bank_col = f'bank_start_A{i+1}'
                if inv_col not in yrc.columns: continue
                yri = yrc[yrc['year'] == y]
                if yri.empty: continue
                row = {
                    'variant': k[1], 'seed': k[2], 'agent': i + 1, 'year': y,
                    'archetype': archetype_of(i, cfg),
                    'weighting': weighting_of(i, cfg),
                    'invest_cost': float(yri[inv_col].mean()),
                    'bank_start':  float(yri[bank_col].mean()) if bank_col in yri.columns else np.nan,
                    'clearing_price':     float(cp_year.get(y, np.nan)),
                    'lag_clearing_price': float(cp_year.get(y - 1, np.nan)) if y > 0 else np.nan,
                }
                # Initial mix slots (designed feature for H2).
                mix = cfg['companies']['initial_mix'][i]
                for j, t in enumerate(['coal', 'gas', 'onshore', 'offshore', 'solar']):
                    row[f'init_{t}'] = float(mix[j])
                rows.append(row)
    return pd.DataFrame(rows)


pay = _per_agent_year_table(DEFAULT_RUNS)
print(f'Per-(agent, year) panel: {len(pay)} rows')
display(pay.head())

In [ ]:
# --- (c) H1 — does carbon price drive green investment? -------------
INV_THRESH = float(np.nanpercentile(pay['invest_cost'], 75)) if len(pay) else 0.0
print(f'Invest threshold (top-quartile invest_cost): {INV_THRESH:.2f} M€')

H1 = pay.dropna(subset=['lag_clearing_price', 'bank_start']).copy()
H1['invests'] = (H1['invest_cost'] > INV_THRESH).astype(int)
arche_dummies = pd.get_dummies(H1['archetype'], prefix='arch', drop_first=True)
H1_X = pd.concat([H1[['lag_clearing_price', 'bank_start', 'year']], arche_dummies], axis=1)
H1_y = H1['invests']

if H1_y.nunique() < 2 or len(H1_y) < 30:
    print('H1 skipped — insufficient label variation or rows.')
else:
    pipe = Pipeline([
        ('scale', StandardScaler()),
        ('lr',    LogisticRegression(penalty='l1', solver='saga', max_iter=2000, C=0.5, random_state=0)),
    ])
    cv_score = cross_val_score(pipe, H1_X, H1_y, cv=StratifiedKFold(n_splits=min(5, H1_y.value_counts().min()), shuffle=True, random_state=0), scoring='roc_auc')
    pipe.fit(H1_X, H1_y)
    coefs = pd.Series(pipe.named_steps['lr'].coef_[0], index=H1_X.columns).sort_values(key=abs, ascending=False)
    print(f'H1 logistic LASSO  —  CV AUC = {cv_score.mean():.3f} ± {cv_score.std():.3f}')
    print('Coefficients (standardised features):')
    display(coefs.to_frame('coef'))

    # Random Forest + partial dependence on lagged clearing price.
    rf = RandomForestClassifier(n_estimators=300, max_depth=None, random_state=0, n_jobs=-1).fit(H1_X, H1_y)
    fi = pd.Series(rf.feature_importances_, index=H1_X.columns).sort_values(ascending=False)
    print('Random forest feature importances:')
    display(fi.to_frame('importance'))

    pd_res = partial_dependence(rf, H1_X, ['lag_clearing_price'], grid_resolution=40)
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(pd_res['grid_values'][0], pd_res['average'][0], lw=2)
    ax.axvline(50, color='red', ls='--', label='theory threshold ≈ 50 €/t')
    ax.set(xlabel='lag clearing price (EUR/t)', ylabel='P(invests | x)',
           title='H1 — RF partial dependence on lagged carbon price')
    ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
# --- (d) H2 — heterogeneity: high-fossil agents transition faster ----
agent_panel = (
    pay.groupby(['variant', 'seed', 'agent', 'archetype', 'weighting'])
       .agg(mean_clearing=('clearing_price', 'mean'),
            total_invest=('invest_cost', 'sum'),
            init_coal=('init_coal', 'first'),
            init_gas=('init_gas', 'first'),
            init_onshore=('init_onshore', 'first'),
            init_offshore=('init_offshore', 'first'),
            init_solar=('init_solar', 'first'))
       .reset_index()
)
if len(agent_panel) > 0 and 'total_invest' in agent_panel.columns:
    med = float(agent_panel['total_invest'].median())
    agent_panel['transitions'] = (agent_panel['total_invest'] > med).astype(int)
    arche_d = pd.get_dummies(agent_panel['archetype'], prefix='arch', drop_first=True)
    H2_X = pd.concat([agent_panel[['mean_clearing', 'init_coal', 'init_gas']], arche_d], axis=1)
    H2_y = agent_panel['transitions']
    if H2_y.nunique() < 2 or len(H2_y) < 8:
        print('H2 skipped — insufficient sample size for cross-validated logit.')
    else:
        pipe2 = Pipeline([('scale', StandardScaler()),
                          ('lr',    LogisticRegression(penalty='l1', solver='saga', max_iter=2000, C=0.5, random_state=0))])
        n_splits_h2 = max(2, min(5, int(H2_y.value_counts().min())))
        cv2 = cross_val_score(pipe2, H2_X, H2_y,
                              cv=StratifiedKFold(n_splits=n_splits_h2, shuffle=True, random_state=0),
                              scoring='roc_auc')
        pipe2.fit(H2_X, H2_y)
        coefs2 = pd.Series(pipe2.named_steps['lr'].coef_[0], index=H2_X.columns).sort_values(key=abs, ascending=False)
        print(f'H2 logistic LASSO — CV AUC = {cv2.mean():.3f} ± {cv2.std():.3f}')
        print('Sign of init_coal positive ⇒ higher-fossil agents transition faster (the theory prediction).')
        display(coefs2.to_frame('coef'))
else:
    print('H2 skipped — agent panel empty.')

**Interpretation.** A positive lag-clearing-price coefficient
in H1 plus a step-up in the RF partial dependence near the theoretical
50 €/t LCOE crossover supports the *carbon price drives green
investment* claim. A positive `init_coal` coefficient in H2 supports
the *high-fossil agents transition faster* claim. The anchor overlay
provides the credibility floor: if realised prices track the
fundamental, the simulation is producing economically-grounded prices,
which feeds RQ1 too.

---

## §R1 — Regulatory Shifts (claim RQ2.a)

> If the loaded sweeps include non-default variants (LRF / MSR / cap
> changes) we run §S1's clustering on each variant separately and
> project everything into the **default-fitted** §S1 PCA basis to make
> the visual comparison direct. Outcomes are summarised with cross-seed
> boxplots and IQR-based outlier flagging.

In [ ]:
variant_keys = sorted({(k[0], k[1]) for k in runs})
variant_keys_overrides = [(s, v) for (s, v) in variant_keys
                           if any(runs[k]['has_overrides'] for k in runs if k[0] == s and k[1] == v)]

if not variant_keys_overrides:
    print('No override variants loaded — §R1 cluster-shift comparison skipped.')
else:
    n_var = len(variant_keys)
    n_cols = min(n_var, 3); n_rows = int(np.ceil(n_var / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5.5 * n_cols, 5 * n_rows), squeeze=False)
    for ax_idx, (sw, vn) in enumerate(variant_keys):
        ax = axes[ax_idx // n_cols, ax_idx % n_cols]
        var_runs = {k: runs[k] for k in runs if k[0] == sw and k[1] == vn}
        v_feat = build_strategy_features(var_runs, conv_starts)
        if v_feat.empty:
            ax.set_axis_off(); continue
        v_X = v_feat[feats].values.astype(float)
        col_means = np.nanmean(v_X, axis=0)
        nan_mask = ~np.isfinite(v_X)
        v_X[nan_mask] = np.take(col_means, np.where(nan_mask)[1])
        v_P = pca.transform(scaler.transform(v_X))
        # Colour by KMeans cluster *as predicted by the default-fitted KMeans* —
        # this turns "where do this variant's agents land?" into "do they fall in
        # default-style clusters or do new clusters emerge?".
        labels_pred = km.predict(scaler.transform(v_X))
        for c in np.unique(labels_pred):
            sel = labels_pred == c
            ax.scatter(v_P[sel, 0], v_P[sel, 1], s=60, alpha=0.85, label=f'default-cluster {c}')
        ax.scatter(v_P[:, 0].mean(), v_P[:, 1].mean(), marker='X', s=260, color='k',
                   edgecolor='white', linewidth=1.5, label='variant centroid')
        ax.set(title=f'{vn}', xlabel='PC1', ylabel='PC2')
        ax.legend(fontsize=7)
    for ax_idx in range(len(variant_keys), n_rows * n_cols):
        axes[ax_idx // n_cols, ax_idx % n_cols].set_axis_off()
    plt.suptitle('§R1 — Strategy positions in §S1 PCA basis, by variant', y=1.02, fontsize=13)
    plt.tight_layout(); plt.show()

In [ ]:
# --- Outcome boxplots with IQR outlier flagging ---------------------
OUT_METRICS = [('mean_clearing_price', lambda r: float(r['yr_df']['clearing_price'].mean()) if 'clearing_price' in r['yr_df'].columns else np.nan),
               ('mean_green_frac',     lambda r: float(np.nanmean([r['ep_df'][f'green_frac_A{i+1}'].iloc[-50:].mean()
                                                                    for i in range(N_AGENTS) if f'green_frac_A{i+1}' in r['ep_df'].columns]))),
               ('total_penalty',       lambda r: float(np.nansum([r['ep_df'][f'penalty_A{i+1}'].iloc[-50:].mean()
                                                                   for i in range(N_AGENTS) if f'penalty_A{i+1}' in r['ep_df'].columns]))),
               ('quality_score',       lambda r: float(r['ep_df']['quality_score'].iloc[-50:].mean()) if 'quality_score' in r['ep_df'].columns else np.nan)]

rows = []
for k, run in runs.items():
    row = {'variant': k[1], 'seed': k[2]}
    for name, fn in OUT_METRICS:
        try: row[name] = fn(run)
        except Exception: row[name] = np.nan
    rows.append(row)
out_df = pd.DataFrame(rows)

fig, axes = plt.subplots(1, len(OUT_METRICS), figsize=(5 * len(OUT_METRICS), 4))
for ax, (name, _) in zip(axes, OUT_METRICS):
    sns.boxplot(data=out_df, x='variant', y=name, ax=ax, showfliers=False)
    sns.stripplot(data=out_df, x='variant', y=name, ax=ax, color='black', size=4, alpha=0.6)
    # IQR-based outlier flag for the within-variant distribution.
    for v in out_df['variant'].unique():
        sub = out_df[out_df['variant'] == v][name].dropna()
        if len(sub) >= 4:
            q1, q3 = sub.quantile([0.25, 0.75]); iqr = q3 - q1
            mask = (sub < q1 - 1.5 * iqr) | (sub > q3 + 1.5 * iqr)
            for v_idx, value in zip(sub.index[mask], sub[mask]):
                ax.annotate('!', xy=(list(out_df['variant'].unique()).index(v), value), color='red', fontsize=12, ha='center')
    ax.set_title(name); ax.tick_params(axis='x', rotation=20)
plt.suptitle('§R1 — Outcome distribution by variant (red ! = IQR outlier seed)', y=1.03)
plt.tight_layout(); plt.show()
display(out_df.groupby('variant').agg(['mean', 'std', 'count']))

---

## §R2 — UDBC Compliance Pathways (claim RQ2.b)

The simulator already classifies each `(agent, year)` cell into one of
five buckets — **U** (under-allocation), **D** (debt-cascade),
**M** (mixed), **B** (bought-into-compliance), **C** (compliant). We
exploit this directly:

* **Multinomial logistic regression** predicting the bucket from
  `bank_start`, `cap`, `clearing_price`, archetype, `year`, and variant.
  **Random Forest** as the non-linear robustness check.
* **Class imbalance:** C dominates → fit with and without **SMOTE**
  oversampling and report the comparison.
* **Transition heatmaps** of year-t → year-t+1 bucket transitions per
  variant.

Per-cell labels live in `udbc_*_total_A{i}` only as *episode totals*,
so we proxy the per-(agent, year) bucket by combining year-log
shortfall and the episode-level bucket totals: a year is `B` if the
agent traded volume on the secondary > 0 and `shortfall ≈ 0`, `C` if
shortfall ≈ 0 and no buying, `U` otherwise (alloc<emiss, no
inherited cf), `D` if there is positive carry-forward, `M` otherwise.

In [ ]:
# Reconstruct per-(run, agent, year) UDBC label from year_log columns.
# Bucket definitions match the in-trainer attribution (see scripts/train.py:1929 onwards).
def _udbc_label_year(yr_row, i):
    sf = float(yr_row.get(f'shortfall_A{i+1}', 0.0))
    bank = float(yr_row.get(f'bank_start_A{i+1}', 0.0))
    alloc = float(yr_row.get(f'alloc_A{i+1}', 0.0))
    emiss = float(yr_row.get(f'emissions_A{i+1}', 0.0))
    sec = float(yr_row.get(f'secondary_net_A{i+1}', 0.0))
    if sf > 1e-6:
        # Non-compliant: classify U / D / M.
        cf = max(0.0, bank)
        under_alloc = alloc < emiss
        has_cf = cf > 1e-6
        if under_alloc and not has_cf: return 'U'
        if (not under_alloc) and has_cf: return 'D'
        return 'M'
    # Compliant: B if net buyer on secondary, else C.
    return 'B' if sec > 1e-6 else 'C'


udbc_rows = []
for k, run in runs.items():
    yr = run['yr_df']; cfg = run['config']
    if yr.empty or 'year' not in yr.columns: continue
    cut = conv_starts.get(k, int(0.8 * len(run['ep_df'])))
    yrc = yr[yr['episode'] >= cut] if 'episode' in yr.columns else yr
    for _, ry in yrc.iterrows():
        for i in range(N_AGENTS):
            udbc_rows.append({
                'variant': k[1], 'seed': k[2], 'agent': i + 1, 'year': int(ry['year']),
                'archetype': archetype_of(i, cfg),
                'weighting': weighting_of(i, cfg),
                'cap': float(ry.get('cap', np.nan)),
                'clearing_price': float(ry.get('clearing_price', np.nan)),
                'bank_start': float(ry.get(f'bank_start_A{i+1}', np.nan)),
                'label': _udbc_label_year(ry, i),
            })
udbc_df = pd.DataFrame(udbc_rows)
print(f'UDBC panel: {len(udbc_df)} (run, agent, year) cells')
print('Class distribution:')
print(udbc_df['label'].value_counts())

In [ ]:
# --- Multinomial logit with / without SMOTE --------------------------
if udbc_df.empty or udbc_df['label'].nunique() < 2:
    print('UDBC modelling skipped — insufficient label variety.')
else:
    feat_cols = ['cap', 'clearing_price', 'bank_start', 'year']
    cat_cols  = ['archetype', 'weighting', 'variant']
    df_m = udbc_df.dropna(subset=feat_cols + ['label']).copy()
    Xm = pd.concat([df_m[feat_cols].astype(float),
                    pd.get_dummies(df_m[cat_cols], drop_first=True)], axis=1)
    ym = df_m['label'].values

    Xtr, Xte, ytr, yte = train_test_split(Xm, ym, test_size=0.25, random_state=0, stratify=ym)
    pipe = Pipeline([('sc', StandardScaler(with_mean=False)),
                     ('lr', LogisticRegression(max_iter=2000, solver='lbfgs', random_state=0))])
    pipe.fit(Xtr, ytr)
    print('Multinomial logit (no SMOTE)  —  test classification report:')
    print(classification_report(yte, pipe.predict(Xte)))

    if _HAS_SMOTE and pd.Series(ytr).value_counts().min() >= 6:
        try:
            from imblearn.pipeline import Pipeline as ImbPipe
            sm_pipe = ImbPipe([('sc', StandardScaler(with_mean=False)),
                               ('sm', SMOTE(random_state=0, k_neighbors=3)),
                               ('lr', LogisticRegression(max_iter=2000, solver='lbfgs', random_state=0))])
            sm_pipe.fit(Xtr, ytr)
            print('Multinomial logit (with SMOTE) — test classification report:')
            print(classification_report(yte, sm_pipe.predict(Xte)))
        except Exception as e:
            print(f'SMOTE step skipped: {e}')
    else:
        print('SMOTE skipped — not enough samples in the minority classes for k-neighbours.')

    # Random Forest robustness + feature importances.
    rfm = RandomForestClassifier(n_estimators=400, random_state=0, n_jobs=-1).fit(Xtr, ytr)
    print('Random forest — test classification report:')
    print(classification_report(yte, rfm.predict(Xte)))
    fi = pd.Series(rfm.feature_importances_, index=Xm.columns).sort_values(ascending=False).head(10)
    display(fi.to_frame('importance'))

In [ ]:
# --- Transition heatmaps year-t → year-t+1 per variant --------------
def _transition_matrix(df_var):
    pairs = []
    for (variant, seed, agent), grp in df_var.groupby(['variant', 'seed', 'agent']):
        grp_sorted = grp.sort_values('year')
        labs = grp_sorted['label'].values
        for a, b in zip(labs[:-1], labs[1:]):
            pairs.append((a, b))
    if not pairs:
        return pd.DataFrame(0, index=UDBC_BUCKETS, columns=UDBC_BUCKETS)
    df_p = pd.DataFrame(pairs, columns=['from', 'to'])
    M = pd.crosstab(df_p['from'], df_p['to'])
    M = M.reindex(index=UDBC_BUCKETS, columns=UDBC_BUCKETS, fill_value=0)
    return M.div(M.sum(axis=1).replace(0, np.nan), axis=0).fillna(0)


variants_present = sorted(udbc_df['variant'].unique())
n_v = len(variants_present)
fig, axes = plt.subplots(1, n_v, figsize=(4.2 * n_v, 4), squeeze=False)
for ax, v in zip(axes[0], variants_present):
    M = _transition_matrix(udbc_df[udbc_df['variant'] == v])
    sns.heatmap(M, annot=True, fmt='.2f', cmap='Blues', vmin=0, vmax=1, ax=ax,
                cbar=(ax is axes[0, -1]))
    ax.set(title=f'{v} — UDBC transitions (rows: from, cols: to)',
           xlabel='to', ylabel='from')
plt.suptitle('§R2 — UDBC year-t → year-t+1 transition probabilities', y=1.04)
plt.tight_layout(); plt.show()

**Interpretation.** Coefficients on `cap` (negative) and
`clearing_price` (positive) drive the model toward U / D states —
that's the regulatory lever turning the screw. The transition heatmap
diagonals tell us which buckets are *sticky*: tighter LRF should
plausibly raise the U → U probability and lower the U → C recovery
rate. Comparing variants side-by-side answers RQ2.b directly.

---

## Summary — Mapping Panels Back to Claims

| Claim | Section | Panels |
|---|---|---|
| RQ1.a — Agents converge by episode N | §C1 | reward + price trajectory with PELT/plateau cutoff |
| RQ1.b — Findings replicate across seeds | §C2 | fan charts + cross-seed CV table |
| RQ3.a — Distinguishable strategic archetypes | §S1 | elbow/silhouette, dendrogram, PCA, archetype × cluster crosstab |
| RQ3.b — Removing ESG component collapses the strategy space | §S2 | within-default contrast + cross-variant PCA + spread/silhouette table |
| RQ3.c — Agents respond rationally to incentives | §S3 | anchor overlay, H1 LASSO + RF partial dependence, H2 logit |
| RQ2.a — Regulatory levers shift strategies and outcomes | §R1 | per-variant PCA panels + outcome boxplots with IQR flags |
| RQ2.b — Stricter regulation changes the *type* of compliance failure | §R2 | multinomial logit (with/without SMOTE) + transition heatmaps |

**Reproducibility.** Every section keys off `(sweep, variant, seed)`
and writes nothing back to disk. Re-run from §1 to refresh.

**Out of scope.** Pathology / autoencoder analyses and the
clearing-price prediction model dropped in v3 of the planning doc are
deliberately not implemented here.